In [1]:
from rdkit import Chem
from rdkit.Chem import rdFingerprintGenerator, rdMolDescriptors, Descriptors, AllChem, DataStructs
import numpy as np
import pandas as pd

import lightgbm as lgb
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score

import warnings
warnings.filterwarnings('ignore', message='X does not have valid feature names')

In [2]:
FP_GEN = rdFingerprintGenerator.GetMorganGenerator(
    radius=2,
    fpSize=2048
)

In [3]:
_SMARTS = {
    'carboxylic_acid': Chem.MolFromSmarts('[CX3](=O)[OX2H]'),
    'pyridine_like': Chem.MolFromSmarts('n1ccccc1'),
    'primary_secondary_amine': Chem.MolFromSmarts('[NX3;H2,H1]'),
    'phenol': Chem.MolFromSmarts('c[OH]'),
    'amide': Chem.MolFromSmarts('C(=O)N')
}

In [4]:
def get_basic_descriptors(mol):
    mw = Descriptors.MolWt(mol)
    logp = Descriptors.MolLogP(mol)
    tpsa = rdMolDescriptors.CalcTPSA(mol)
    hbd = rdMolDescriptors.CalcNumHBD(mol)
    hba = rdMolDescriptors.CalcNumHBA(mol)
    rot = rdMolDescriptors.CalcNumRotatableBonds(mol)
    fracsp3 = rdMolDescriptors.CalcFractionCSP3(mol)
    narom = rdMolDescriptors.CalcNumAromaticRings(mol)
    nheavy = mol.GetNumHeavyAtoms()
    formal_charge = Chem.GetFormalCharge(mol)

    nN = sum(1 for a in mol.GetAtoms() if a.GetAtomicNum() == 7)
    nO = sum(1 for a in mol.GetAtoms() if a.GetAtomicNum() == 8)

    return np.array([
        mw, logp, tpsa, hbd, hba, rot, fracsp3, narom, nheavy, formal_charge, nN, nO
    ], dtype=float)

In [5]:
def get_flags(mol: Chem.Mol):
    flags = []
    for name, pattern in _SMARTS.items():
        flags.append(1 if mol.HasSubstructMatch(pattern) else 0)

    return np.array(flags, dtype=np.int8)

In [6]:
def get_gaisteger_stats(mol: Chem.Mol):
    AllChem.ComputeGasteigerCharges(mol)
    charges = np.array([float(a.GetProp('_GasteigerCharge')) for a in mol.GetAtoms()], dtype=float)

    max_q = np.max(charges)
    min_q = np.min(charges)
    sum_positive = np.sum(charges[charges > 0])
    sum_negative = np.sum(charges[charges < 0])

    return np.array([
        max_q, min_q, sum_positive, sum_negative
    ], dtype=float)

In [7]:
def get_advanced_chemistry(mol: Chem.Mol):
    num_F = sum(1 for a in mol.GetAtoms() if a.GetAtomicNum() == 9)
    num_Cl = sum(1 for a in mol.GetAtoms() if a.GetAtomicNum() == 17)
    num_Br = sum(1 for a in mol.GetAtoms() if a.GetAtomicNum() == 35)

    labute_asa = rdMolDescriptors.CalcLabuteASA(mol)
    tpsa = rdMolDescriptors.CalcTPSA(mol)
    p_surface = tpsa / labute_asa if labute_asa > 0 else 0

    complexity = Descriptors.BertzCT(mol)

    is_perfluoro = 1 if num_F > 4 else 0 

    return np.array([
        num_F, num_Cl, num_Br, p_surface, complexity, is_perfluoro, labute_asa
    ])

In [8]:
def prepare_features(input_df: pd.DataFrame):
    features = []
    for _, row in input_df.iterrows():
        mol1 = Chem.MolFromSmiles(row['SMILES1'])
        mol2 = Chem.MolFromSmiles(row['SMILES2'])

        desc1 = get_basic_descriptors(mol1)
        desc2 = get_basic_descriptors(mol2)

        adv1 = get_advanced_chemistry(mol1)
        adv2 = get_advanced_chemistry(mol2)

        halogen_bond_proxy = adv1[0] * desc2[10] + adv2[0] * desc1[10]
        size_mismatch = max(desc1[0], desc2[0]) / (min(desc1[0], desc2[0]) + 1e-6)

        # ------------ Фингерпринты ----------------------------- #

        fp1_bin = np.array(FP_GEN.GetCountFingerprintAsNumPy(mol1) > 0, dtype=np.int8)
        fp2_bin = np.array(FP_GEN.GetCountFingerprintAsNumPy(mol2) > 0, dtype=np.int8)

        diff = np.abs(fp1_bin - fp2_bin)
        summ = fp1_bin + fp2_bin
        inter = fp1_bin * fp2_bin
        fp_features = np.concatenate([diff, summ, inter])

        # ------------ Новые дескрипторы ----------------------------- #

        desc_diff = np.abs(desc1 - desc2)
        desc_sum = desc1 + desc2
        hbd_hba_comp = np.array([
            desc1[3] * desc2[4],
            desc2[3] *  desc1[4]
        ], dtype=float)

        # ------------ Флаги ----------------------------- #
        
        flags1 = get_flags(mol1)
        flags2 = get_flags(mol2)

        acid_idx = list(_SMARTS.keys()).index('carboxylic_acid')
        base_idx = list(_SMARTS.keys()).index('pyridine_like')
        acid_base_flags = np.array([
            int(flags1[acid_idx] == 1 and (flags2[base_idx] == 1 or flags2[list(_SMARTS.keys()).index('primary_secondary_amine')] == 1)),
            int(flags2[acid_idx] == 1 and (flags1[base_idx] == 1 or flags1[list(_SMARTS.keys()).index('primary_secondary_amine')] == 1))
        ], dtype=np.int8)

        # ------------ Заряды Гастайгера ----------------------------- #

        g1 = get_gaisteger_stats(mol1)
        g2 = get_gaisteger_stats(mol2)

        g_diff = np.abs(g1 - g2)
        g_sum = g1 + g2

        mfp1 = FP_GEN.GetFingerprint(mol1)
        mfp2= FP_GEN.GetFingerprint(mol2)
        tanimoto = np.array([
            DataStructs.TanimotoSimilarity(mfp1, mfp2)
        ])

        # ------------ Собираем все числовые фичи в кучу ----------------------------- #

        new_numeric = np.concatenate([
            desc1, desc2, desc_diff, desc_sum,
            hbd_hba_comp,
            flags1.astype(float), flags2.astype(float),
            acid_base_flags.astype(float),
            g1, g2, g_diff, g_sum,
            tanimoto,
            np.array([halogen_bond_proxy, size_mismatch]),
            adv1, adv2 
        ])

        feature_vector = np.concatenate([fp_features.astype(np.int8), new_numeric.astype(float)])

        features.append(feature_vector)

    return np.array(features)

In [ ]:
data = pd.read_csv("train_dataset/train.csv")
test_data = pd.read_csv('test.csv')
sample_sub = pd.read_csv('sample_submission.csv')

print('Генерация признаков...')
X = prepare_features(data)
y = data['result'].values

X_test = prepare_features(test_data)

print('done')

Генерация признаков...
done


In [55]:
skf = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

test_predicts = np.zeros(len(test_data))
oof_predicts = np.zeros(len(data))

In [17]:
lgb_params = {
    'objective': 'binary',
    'metric': 'binary_logloss',
    'boosting_type': 'gbdt',
    'learning_rate': 0.03,
    'num_leaves': 63,
    'class_weight': "balanced",
    'random_state': 41,
    'n_estimators': 1000,
    'verbosity': 0
}

In [57]:
train_losses = []
val_losses = []

for fold, (train_idx, val_idx) in enumerate(skf.split(X, y)):
    print(f"{'-' * 10} Фолд {fold + 1} {'-' * 10}")
    X_train, y_train = X[train_idx], y[train_idx]
    X_val, y_val = X[val_idx], y[val_idx]

    model = lgb.LGBMClassifier(**lgb_params)
    model.fit(
        X_train, y_train,
        eval_set=[(X_val, y_val)],
        callbacks=[lgb.early_stopping(stopping_rounds=100)]
    )

    oof_predicts[val_idx] = model.predict_proba(X_val)[:, 1]
    test_predicts += model.predict_proba(X_test)[:, 1] / skf.n_splits

best_threshold = 0.5
best_score = 0.0

for thresh in np.linspace(0.3, 1, 61):
    y_pred_thresh = (oof_predicts > thresh).astype(float)

    acc_0 = accuracy_score(y[y == 0], y_pred_thresh[y == 0])
    acc_1 = accuracy_score(y[y == 1], y_pred_thresh[y == 1])

    current_weighted_score = 0.7 * acc_0 + 0.3 * acc_1

    if current_weighted_score > best_score:
        best_score = current_weighted_score
        best_threshold = thresh

print(f"Лучший порог:        {best_threshold=}")
print(f"Валидационный скор:  {best_score=:.4f}")

---------- Фолд 1 ----------
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[424]	valid_0's binary_logloss: 0.0911158
---------- Фолд 2 ----------
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[387]	valid_0's binary_logloss: 0.0984053
---------- Фолд 3 ----------
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[358]	valid_0's binary_logloss: 0.122994
---------- Фолд 4 ----------
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[524]	valid_0's binary_logloss: 0.0853963
---------- Фолд 5 ----------
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[436]	valid_0's binary_logloss: 0.103681
Лучший порог:        best_threshold=0.9533333333333334
Валидационный скор:  best_score=0.9462


In [59]:
sample_sub['result'] = (test_predicts >= best_threshold).astype(int)
sample_sub.to_csv('submission.csv', index=False)